# 12 — Overfitting, weight decay, and dropout

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/012-overfitting-in-deep-learning.ipynb)

A model with more knobs than data will memorise the noise it was shown. Here you fit a degree-9 polynomial to twelve noisy points, add one penalty term to calm it down, and then break it on purpose by leaving dropout switched on at prediction time. Every number is computed with NumPy, so you can rerun each cell on a fresh Colab runtime.

## 1. The numbers: one fit with a penalty, one without

The data is a sine wave sampled at twelve points, with noise added. A degree-9 polynomial has ten coefficients, so it has almost as many knobs as data points and can pass through every one of them. Penalising large weights (`lam`) trades a little training accuracy for a much better answer between and beyond the points.

In [ ]:
import numpy as np                                     # arrays and linear algebra

rng = np.random.default_rng(0)                         # fixed seed: same numbers on every run
x_train = np.linspace(0, 1, 12)                        # 12 inputs, evenly spread over [0, 1]
y_train = np.sin(2 * np.pi * x_train) + 0.15 * rng.normal(size=12)   # true curve plus noise
x_test = np.linspace(0, 1, 200)                        # a dense grid used only for grading
y_test = np.sin(2 * np.pi * x_test)                    # the clean curve: the test set has no noise

def design(x):                                         # turn inputs into 10 columns: x^0 ... x^9
    return np.vander(x, 10, increasing=True)           # row i holds the ten powers of x[i]

def fit(x, y, lam):                                    # ridge regression: least squares plus a penalty
    X = design(x)                                      # the 12 x 10 feature matrix
    A = X.T @ X + lam * np.eye(10)                     # normal equations, lam shrinks big weights
    return np.linalg.solve(A, X.T @ y)                 # solve A w = X^T y for the ten weights

def rmse(w, x, y):                                     # root mean squared error, in the units of y
    return np.sqrt(np.mean((design(x) @ w - y) ** 2))  # average squared miss, then square root

w_plain = fit(x_train, y_train, 0.0)                   # no penalty: chase the 12 noisy points
w_ridge = fit(x_train, y_train, 1e-6)                  # a whisper of a penalty on weight size
print("largest |w|, no penalty :", np.abs(w_plain).max().round(1))   # 34061: the fit wiggles hard
print("largest |w|, penalty   :", np.abs(w_ridge).max().round(1))    # 23.4: already calm
print("train RMSE, no penalty:", rmse(w_plain, x_train, y_train).round(3))   # 0.005
print("test  RMSE, no penalty:", rmse(w_plain, x_test, y_test).round(3))     # 0.111
print("train RMSE, penalty   :", rmse(w_ridge, x_train, y_train).round(3))   # 0.055
print("test  RMSE, penalty   :", rmse(w_ridge, x_test, y_test).round(3))     # 0.091

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import numpy as np</code> — loads NumPy, which Colab already has.</li><li><code>rng = np.random.default_rng(0)</code> — a random generator with a fixed seed, so results repeat.</li><li><code>x_train = ...</code> — twelve input values spread evenly from 0 to 1.</li><li><code>y_train = ...</code> — the true sine plus Gaussian noise of size 0.15.</li><li><code>x_test</code> / <code>y_test</code> — a dense, noise-free grid used only to grade the model.</li><li><code>def design(x)</code> — builds the ten powers of each input, one column per power.</li><li><code>np.vander(x, 10, increasing=True)</code> — NumPy's helper for exactly that matrix.</li><li><code>def fit(x, y, lam)</code> — the whole training procedure in three lines.</li><li><code>A = X.T @ X + lam * np.eye(10)</code> — the normal equations, with <code>lam</code> added to the diagonal.</li><li><code>np.linalg.solve(A, X.T @ y)</code> — solves for the ten weights exactly, no iteration.</li><li><code>def rmse(...)</code> — one number for error, in the same units as <code>y</code>.</li><li><code>w_plain</code> / <code>w_ridge</code> — the same fit at <code>lam = 0</code> and <code>lam = 1e-6</code>.</li><li><code>print(...)</code> — the biggest weight and the two errors for each model.</li></ul>

</details>

## 2. Step by step: walk up the penalty

Instead of guessing a good `lam`, sweep it. Each pass refits the same ten weights and records two errors: the one on the points the model saw, and the one on the clean curve. Watch the training error rise while the test error falls.

In [ ]:
lambdas = [0.0, 1e-12, 1e-9, 1e-6, 1e-4, 1e-2, 1.0]     # seven settings of the one knob

for lam in lambdas:                                    # one refit per setting
    w = fit(x_train, y_train, lam)                     # same data, different penalty
    train_err = rmse(w, x_train, y_train)              # error on the points it saw
    test_err = rmse(w, x_test, y_test)                 # error on the clean curve
    print(f"lam={lam:8.0e}  train={train_err:.3f}  test={test_err:.3f}")   # one row per iteration

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>lambdas = [...]</code> — the penalty sizes to try, from none to quite strong.</li><li><code>for lam in lambdas:</code> — one loop iteration per penalty.</li><li><code>w = fit(...)</code> — refit the ten weights at this penalty.</li><li><code>train_err</code> — root mean squared error on the twelve noisy training points.</li><li><code>test_err</code> — root mean squared error on the clean grid.</li><li><code>print(f"...")</code> — an f-string formats each number into one line of the table.</li></ul>

</details>

## 3. Plot: training and test error against the penalty

On a log scale for `lam`, the two curves cross: past a point, extra penalty starts to hurt both. The gap between them is the part of the error that comes from memorising noise rather than learning the curve.

In [ ]:
import matplotlib.pyplot as plt                        # the plotting library

grid = np.logspace(-12, 1, 26)                         # penalty sizes from 1e-12 up to 10
train_curve, test_curve = [], []                       # one error per penalty size

for lam in grid:                                       # walk up the penalty axis
    w = fit(x_train, y_train, lam)                     # refit at this penalty
    train_curve.append(rmse(w, x_train, y_train))      # remember the training error
    test_curve.append(rmse(w, x_test, y_test))         # remember the test error

plt.figure(figsize=(7, 3.5))                           # a wide, short figure
plt.semilogx(grid, train_curve, "o-", label="train")   # training error, dots at each sample
plt.semilogx(grid, test_curve, "s-", label="test")     # test error, squares
plt.xlabel("weight decay lambda (log scale)")          # the knob being turned
plt.ylabel("RMSE")                                     # error, in the units of y
plt.legend()                                           # name the two curves
plt.title("a little lambda closes the train-test gap") # the takeaway
plt.show()                                             # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>import matplotlib.pyplot as plt</code> — Colab ships matplotlib.</li><li><code>grid = np.logspace(...)</code> — 26 penalty sizes, evenly spaced on a log scale.</li><li><code>train_curve, test_curve = [], []</code> — two empty lists to fill in the loop.</li><li><code>for lam in grid:</code> — one refit per penalty size.</li><li><code>w = fit(...)</code> — the model at this penalty.</li><li><code>train_curve.append(...)</code> / <code>test_curve.append(...)</code> — store both errors per point.</li><li><code>plt.semilogx(...)</code> — plot with a logarithmic x axis; the rest only labels the figure.</li><li><code>plt.title(...)</code> — states the one thing the reader should take away.</li></ul>

</details>

## 4. Interactive: choose the penalty yourself

The slider sets the exponent of ten, so −12 means almost no penalty and 1 means a lot. The black dots are the noisy training data, the grey line is the curve we want, and the red line is the fitted model. At the left end the red line dives out of the frame to hit every dot; drag right and it settles onto the grey curve.

In [ ]:
from ipywidgets import interact                        # sliders that rerun a function

def show(log_lam=0.0):                                 # the slider supplies log10 of the penalty
    lam = 10.0 ** log_lam                              # turn the exponent into a penalty size
    w = fit(x_train, y_train, lam)                     # refit at this penalty
    xs = np.linspace(0, 1, 200)                        # a smooth grid for drawing the curve
    plt.figure(figsize=(6, 3.5))                       # a fresh figure on every drag
    plt.plot(xs, np.sin(2 * np.pi * xs), color="grey", label="true curve")      # the target
    plt.plot(xs, design(xs) @ w, color="tab:red", label="degree-9 fit")         # the model
    plt.scatter(x_train, y_train, color="black", zorder=3, label="noisy data")  # what it saw
    plt.ylim(-1.5, 1.5)                                # fixed axes, so nothing rescales
    plt.legend(loc="upper right")                      # name all three layers
    tr = rmse(w, x_train, y_train)                     # training error readout
    te = rmse(w, x_test, y_test)                       # test error readout
    plt.title(f"lambda={lam:.0e}   train={tr:.3f}   test={te:.3f}")   # live numbers
    plt.show()                                         # draw the frame

interact(show, log_lam=(-12.0, 1.0, 0.5))              # slider: start, end, step of the exponent

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>from ipywidgets import interact</code> — the widget helper, preinstalled on Colab.</li><li><code>def show(log_lam=0.0)</code> — the function that redraws one frame; the default is the start value.</li><li><code>lam = 10.0 ** log_lam</code> — the slider moves on a log scale, so the penalty spans many orders.</li><li><code>w = fit(...)</code> — refit from scratch on every drag.</li><li><code>xs = np.linspace(...)</code> — 200 points for a smooth curve.</li><li><code>plt.plot(xs, np.sin(...))</code> — the true curve, in grey.</li><li><code>plt.plot(xs, design(xs) @ w)</code> — the model's predictions across the grid.</li><li><code>plt.scatter(...)</code> — the twelve noisy training points.</li><li><code>plt.ylim(-1.5, 1.5)</code> — fixed vertical range; a wild fit visibly leaves the frame.</li><li><code>tr</code> / <code>te</code> — the two errors shown in the title as a readout.</li><li><code>interact(show, log_lam=(...))</code> — attaches the slider with its range and step.</li></ul>

</details>

## 5. When it breaks: dropout left on at inference

Dropout randomly zeroes half of a layer's inputs during training, which is a strong regulariser. It is only valid during training: at prediction time it must be switched off, or the keep probability folded into the weights. Here a trained layer is asked the same twelve questions sixty times with dropout still on, so identical inputs get different answers.

In [ ]:
rng2 = np.random.default_rng(1)                        # a second generator for the dropout demo
n_feat = 40                                            # this layer has 40 inputs
w_dense = rng2.normal(size=n_feat) / np.sqrt(n_feat)   # the weights a trained layer would hold
x_query = rng2.uniform(size=(12, n_feat))              # 12 new inputs, 40 features each
clean = x_query @ w_dense                              # the prediction with dropout correctly off

keep = 0.5                                             # dropout keeps each input half the time
draws = []                                             # one prediction vector per inference pass
for t in range(60):                                    # 60 passes, as if 60 users asked
    mask = rng2.random(x_query.shape) < keep           # True means this feature survives
    draws.append((x_query * mask / keep) @ w_dense)    # inverted dropout: survivors scaled by 1/keep
draws = np.array(draws)                                # shape (60 passes, 12 inputs)

print("spread of clean predictions  :", clean.std().round(3))    # 0.150
print("spread with dropout left on  :", draws.std().round(3))     # 0.527: about 3.5x noisier
print("worst swing on one input     :", (draws.max(0) - draws.min(0)).max().round(2))   # 2.91

plt.figure(figsize=(7, 3.5))                           # a wide figure
for k in range(12):                                    # one vertical cluster per input
    plt.scatter(np.full(60, k), draws[:, k], s=8, color="tab:blue", alpha=0.25)   # 60 answers
plt.plot(range(12), clean, "o", color="tab:red", label="dropout off")   # the intended answer
plt.xlabel("input number")                             # which of the twelve inputs
plt.ylabel("prediction")                               # the model's output
plt.legend()                                           # name the red dots
plt.title("dropout at inference: same input, different answer")   # the takeaway
plt.show()                                             # render the figure

<details>
<summary><strong>Line by line</strong> (click to open)</summary>

<ul><li><code>rng2 = ...</code> — a separate generator so this section does not disturb section 1.</li><li><code>n_feat = 40</code> — the width of the toy layer.</li><li><code>w_dense = ...</code> — fixed weights, scaled so predictions stay around 0.15 in size.</li><li><code>x_query</code> — twelve new inputs, forty features each.</li><li><code>clean = x_query @ w_dense</code> — the answer when dropout is off, which is what we want.</li><li><code>keep = 0.5</code> — the probability that any input survives a dropout mask.</li><li><code>for t in range(60):</code> — sixty independent prediction passes.</li><li><code>mask = ... &lt; keep</code> — a random True/False for every input feature.</li><li><code>(x_query * mask / keep) @ w_dense</code> — zero the dropped features, rescale the rest.</li><li><code>print(...)</code> — shows the predictions are far noisier than the clean ones.</li><li><code>plt.scatter(...)</code> — one dot per pass, so the vertical spread is visible.</li><li><code>plt.plot(range(12), clean, "o")</code> — the single answer we should have got.</li></ul>

</details>

## Your turn

Set the slider to −12 and then to 1: which of the two curves in section 3 does the red line trace at each end? Then change `keep` in section 5 to 0.9 and predict, before rerunning, whether the cloud of dots gets wider or narrower.